

===== UNIT II =====

LAB EXPERIMENTS


In [3]:
!pip install -q openai pandas

from google.colab import userdata
from openai import OpenAI

MAX_CALLS = 40
call_count = 0

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEYS"),
)
MODEL = "llama-3.3-70b-versatile"


def ask_llm(prompt, system=None, temperature=0.3):
    """Send a prompt to Groq. Raises if daily call cap exceeded."""
    global call_count
    call_count += 1
    if call_count > MAX_CALLS:
        raise RuntimeError(f"Lab limit reached ({MAX_CALLS} calls). Stop and save your work.")
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

print("Setup complete. Model:", MODEL)


Setup complete. Model: llama-3.3-70b-versatile


In [7]:
# 1. Few-Shot vs. Zero-Shot Comparison: Design and execute a zero-shot prompt and a few-shot prompt (with 2-3 exemplar input-output pairs) for a chosen text task (e.g., sentiment classification or translation); compare outputs for accuracy, consistency, and adherence to examples.

#  — Zero-shot vs few-shot preview (run with instructor)
review = "The API key setup was confusing at first."

# ---------------- ZERO-SHOT PROMPT ----------------
zero_shot = f"""
Classify the sentiment of the following review as Positive, Negative, or Neutral.

Review: "{review}"

Return only the sentiment label.
"""

# ---------------- FEW-SHOT PROMPT ----------------
few_shot = f"""
Classify the sentiment of the following review as Positive, Negative, or Neutral.

Examples:

Review: "The lab was very interesting and useful."
Sentiment: Positive

Review: "The software crashed several times."
Sentiment: Negative

Review: "The class started at 10 AM."
Sentiment: Neutral

Now classify:

Review: "{review}"
Sentiment:
"""

# ---------------- EXECUTE ----------------
print("Review:", review)

zero_output = ask_llm(zero_shot).strip()
few_output = ask_llm(few_shot).strip()

print("\n===== ZERO-SHOT OUTPUT =====")
print(zero_output)

print("\n===== FEW-SHOT OUTPUT =====")
print(few_output)

# ---------------- EXPECTED ANSWER ----------------
expected = "Negative"

# ---------------- COMPARISON ----------------
print("\n===== COMPARISON =====")

print("Expected sentiment:", expected)

print("\nAccuracy:")
print("Zero-shot:", "Correct" if expected.lower() in zero_output.lower() else "Incorrect")
print("Few-shot:", "Correct" if expected.lower() in few_output.lower() else "Incorrect")

print("\nConsistency:")
print("Zero-shot follows the requested sentiment labels.")
print("Few-shot follows the sentiment labels demonstrated in the examples.")

print("\nAdherence to Examples:")
print("Zero-shot: No examples were provided.")
print("Few-shot: Uses the 3 input-output examples as guidance.")

print("\nConclusion:")
print("Few-shot prompting generally improves consistency and adherence to the required format.")

Review: The API key setup was confusing at first.

===== ZERO-SHOT OUTPUT =====
Negative

===== FEW-SHOT OUTPUT =====
Sentiment: Negative

Although the review doesn't express extreme dissatisfaction, it does mention a negative experience ("confusing"), which indicates a slightly negative sentiment. The phrase "at first" suggests that the issue might have been resolved, but the initial confusion is still worth noting as a negative aspect.

===== COMPARISON =====
Expected sentiment: Negative

Accuracy:
Zero-shot: Correct
Few-shot: Correct

Consistency:
Zero-shot follows the requested sentiment labels.
Few-shot follows the sentiment labels demonstrated in the examples.

Adherence to Examples:
Zero-shot: No examples were provided.
Few-shot: Uses the 3 input-output examples as guidance.

Conclusion:
Few-shot prompting generally improves consistency and adherence to the required format.


In [5]:
# 2. Role-Based & Negative Prompting: Craft a role-based prompt to establish a specific persona (e.g., "You are a financial advisor..."); then create a negative prompt to suppress undesired content (e.g., "Do not mention any brand names"); evaluate how each influences the model's response.

prompt1 = """
You are a financial advisor for college students. Explain the difference between saving and investing in 100–120 words. Use simple language and provide one practical example."""

prompt2 = """
You are a financial advisor for college students. Explain the difference between saving and investing in 100–120 words. Use simple language and provide one practical example.

Do not mention any company or brand names.
Do not guarantee profits or returns.
Do not use technical financial jargon."""

print("=== Prompt 1 ===")
output=ask_llm(prompt1)
print(output)
print(prompt1)

print("\n=== Prompt 2 ===")
print(prompt2)
output=ask_llm(prompt2)
print(output)



=== Prompt 1 ===
As a college student, it's essential to understand the difference between saving and investing. Saving means putting money aside in a safe place, like a bank account, to use later. Investing means using your money to try to grow it over time, often by buying stocks, bonds, or other assets. For example, you could save $1,000 in a bank account or invest it in a stock that might grow to $1,200 over a year. Investing carries more risk, but can also lead to higher rewards. Consider your goals and timeline before deciding which option is best for you.

You are a financial advisor for college students. Explain the difference between saving and investing in 100–120 words. Use simple language and provide one practical example.

=== Prompt 2 ===

You are a financial advisor for college students. Explain the difference between saving and investing in 100–120 words. Use simple language and provide one practical example.

Do not mention any company or brand names.
Do not guarantee 

In [6]:
# 3. Constraint Specification & Iterative Refinement: Select an open-ended task (e.g., summarizing a technical article); issue a basic prompt; identify failures in length or format; refine the prompt by adding explicit constraints (word count, bullet format, etc.); document improvements over two refinement cycles.

prompt1 = """
Summarize the following article about Artificial Intelligence.
"""

prompt2 = """
Summarize the following Artificial Intelligence article in 80–100 words. Include the main idea, three key applications, and one major benefit. Use a formal academic tone.
"""

prompt3 = """
Summarize the following Artificial Intelligence article in exactly five bullet points. Each bullet should contain 12–15 words. Include:
- Definition of AI
- Two applications
- One benefit
- A concluding statement
"""

print("=== Prompt 1 ===")
print(prompt1)
output = ask_llm(prompt1)
print(output)

print("\n=== Prompt 2 ===")
print(prompt2)
output = ask_llm(prompt2)
print(output)

print("\n=== Prompt 3 ===")
print(prompt3)
output = ask_llm(prompt3)
print(output)


=== Prompt 1 ===

Summarize the following article about Artificial Intelligence.

There is no article provided. Please share the article about Artificial Intelligence, and I will be happy to summarize it for you.

=== Prompt 2 ===

Summarize the following Artificial Intelligence article in 80–100 words. Include the main idea, three key applications, and one major benefit. Use a formal academic tone.

Unfortunately, I don't see an article provided. Please share the Artificial Intelligence article, and I'll be happy to summarize it in 80-100 words, covering the main idea, three key applications, and one major benefit, all while maintaining a formal academic tone.

=== Prompt 3 ===

Summarize the following Artificial Intelligence article in exactly five bullet points. Each bullet should contain 12–15 words. Include:
- Definition of AI
- Two applications
- One benefit
- A concluding statement

* Artificial Intelligence refers to intelligent machines performing tasks.
* AI applications incl